<a href="https://colab.research.google.com/github/Disha092006/Flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Disha092006/Flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
login(token=token, add_to_git_credential=False)

print("Hugging Face login successful")

Hugging Face login successful


### 1. Unit of analysis + time window

One row represents one content item for one client on one report date. I will use the March 2026 partition as the mid-panel time window for verification and contract development. June 2026 is treated as a sealed final month rather than being used to design the contract.

In [8]:
import pandas as pd
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=token
)

march_df = pd.read_parquet(file_path)

print("Rows:", len(march_df))
print("Columns:", len(march_df.columns))
print("Columns:")
print(march_df.columns.tolist())

march_df.head()
# Load April 2026 for the future outcome
april_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=token
)

april_df = pd.read_parquet(april_path)

print("April rows:", len(april_df))
print("April columns:", len(april_df.columns))
# Create an April outcome for each content item
april_outcome = (
    april_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        april_impressions=("gsc_impressions", "sum"),
        april_clicks=("gsc_clicks", "sum")
    )
)

# Define the future label:
# 1 = no GSC clicks observed in April
# 0 = at least one GSC click observed in April
april_outcome["is_declining_label"] = (
    april_outcome["april_clicks"].fillna(0) == 0
).astype(int)

print("April content outcomes:", len(april_outcome))
print("\nLabel distribution:")
print(april_outcome["is_declining_label"].value_counts())

Rows: 9841378
Columns: 30
Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
April rows: 10424730
April columns: 30
April content outcomes: 362172

Label distribution:
is_declining_label
1    294340
0     67832
Name: count, dtype: int64


### 2. Fields: feature / label / context / excluded

**Features:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`, `sessions_ai`, and `scroll_events`.

**Label:** This warehouse table contains daily performance measures, but it does not contain the final `is_declining_label` from the earlier task. Therefore, the label will be created later from an explicitly defined future performance window rather than using future information as a feature.

**Context:** `report_date`, `client_hash_id`, `content_hash_id`, `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, and `ga4_data_available`.

**Excluded:** `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, and `ai_other` are excluded from the first five-feature frame because they are detailed AI-source breakdowns and are not needed for the initial contract.

In [9]:
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"
]

context_cols = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available"
]

excluded_cols = [
    "ai_chatgpt",
    "ai_perplexity",
    "ai_gemini",
    "ai_copilot",
    "ai_claude",
    "ai_meta",
    "ai_other"
]

print("Feature fields:", len(feature_cols))
print("Context fields:", len(context_cols))
print("Excluded fields:", len(excluded_cols))

print("\nFeature fields:")
print(feature_cols)

print("\nContext fields:")
print(context_cols)

print("\nExcluded fields:")
print(excluded_cols)

Feature fields: 15
Context fields: 7
Excluded fields: 7

Feature fields:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'scroll_events']

Context fields:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available']

Excluded fields:
['ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other']


### 3. Verify it with queries

The March 2026 partition is used as the mid-panel verification slice. I will verify the row grain, the slice size and date range, and data availability. The availability check uses explicit boolean `IS TRUE` conditions rather than treating missing values as available.

In [10]:
# Query 1 — grain check
grain_check = march_df.groupby(
    ["report_date", "client_hash_id", "content_hash_id"]
).size()

print("Total rows:", len(march_df))
print("Unique date-client-content combinations:", len(grain_check))
print("Maximum rows per date-client-content combination:", grain_check.max())

print("\nGrain verified:", grain_check.max() == 1)
# Query 2 — slice row count and date span
print("March 2026 row count:", len(march_df))
print("Minimum report date:", march_df["report_date"].min())
print("Maximum report date:", march_df["report_date"].max())
# Query 3 — availability check using explicit boolean conditions
available_mask = (
    march_df["gsc_data_available"].eq(True) |
    march_df["ga4_data_available"].eq(True)
)

available_rows = march_df.loc[available_mask]

print("Rows with GSC or GA4 data available:", len(available_rows))
print("Rows surviving availability check:", len(available_rows))

available_rows[
    [
        "report_date",
        "content_hash_id",
        "gsc_data_available",
        "ga4_data_available"
    ]
].head()

Total rows: 9841378
Unique date-client-content combinations: 9841378
Maximum rows per date-client-content combination: 1

Grain verified: True
March 2026 row count: 9841378
Minimum report date: 2026-03-01
Maximum report date: 2026-03-31
Rows with GSC or GA4 data available: 3660680
Rows surviving availability check: 3660680


,report_date,content_hash_id,gsc_data_available,ga4_data_available
0,2026-03-01,content_b7e512995f79d5a6,True,None
1,2026-03-01,content_05597932fe4da067,True,None
2,2026-03-01,content_7a105f548d9c6916,True,None
3,2026-03-01,content_905aa32a0230694e,True,None
4,2026-03-01,content_a3ea9792f793ec72,True,None


### 4A. Five-feature frame

I will use five features that are measurable at the decision moment and have a clear operational meaning:

1. `gsc_impressions` — knowable at the decision moment because it is already recorded in the current reporting window.
2. `gsc_clicks` — knowable at the decision moment because it is already recorded in the current reporting window.
3. `gsc_avg_position` — knowable at the decision moment because it is calculated from observed GSC performance.
4. `ga4_sessions` — knowable at the decision moment because it is recorded from observed GA4 activity.
5. `ga4_engaged_sessions` — knowable at the decision moment because it is recorded from observed GA4 activity.

These features describe observed performance at the decision moment. Future performance is reserved for the label and is not used as an input feature.
### 4B. Deliberate leakage trap

To demonstrate leakage, I deliberately used the future April `is_declining_label` itself as an input feature. This information would not be available at the March decision moment. The model achieved a ROC-AUC of 1.0, showing that the leaked feature makes the prediction artificially perfect. The leaked feature is removed from the honest five-feature set.

In [11]:
# Five honest features from the March decision window
five_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

# Keep only March information available at the decision moment
march_features = march_df[
    ["client_hash_id", "content_hash_id"] + five_features
].copy()

# Join the future April outcome
contract_df = march_features.merge(
    april_outcome[
        ["client_hash_id", "content_hash_id", "is_declining_label"]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Contract rows:", len(contract_df))
print("Columns:", contract_df.columns.tolist())

print("\nFive-feature availability:")
print(
    contract_df[five_features]
    .notna()
    .mean()
    .mul(100)
    .round(2)
    .astype(str)
    .add("%")
)

print("\nFuture label distribution:")
print(contract_df["is_declining_label"].value_counts())

contract_df.head()

Contract rows: 9841377
Columns: ['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'is_declining_label']

Five-feature availability:
gsc_impressions         100.0%
gsc_clicks              100.0%
gsc_avg_position        36.69%
ga4_sessions            69.33%
ga4_engaged_sessions    69.33%
dtype: object

Future label distribution:
is_declining_label
1    8021531
0    1819846
Name: count, dtype: int64


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,is_declining_label
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,NaN,NaN,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,NaN,NaN,1
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,NaN,NaN,0
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,NaN,NaN,1
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,NaN,NaN,0


In [12]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Deliberate leakage:
# use the future April outcome itself as a feature
leak_df = contract_df[["is_declining_label"]].copy()

leak_df["leaked_feature"] = leak_df["is_declining_label"]

X = leak_df[["leaked_feature"]]
y = leak_df["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leak_model = RandomForestClassifier(
    n_estimators=50,
    random_state=42
)

leak_model.fit(X_train, y_train)

leak_pred = leak_model.predict_proba(X_test)[:, 1]
leak_auc = roc_auc_score(y_test, leak_pred)

print("Deliberately leaked feature: future is_declining_label")
print("Leakage ROC-AUC:", round(leak_auc, 4))
print("The leaked feature is removed from the honest five-feature set.")

Deliberately leaked feature: future is_declining_label
Leakage ROC-AUC: 1.0
The leaked feature is removed from the honest five-feature set.


### 5. Self-check

- Five plain-words contract answers are documented.
- Exactly three verification queries are shown for March 2026.
- Availability is checked using explicit boolean conditions.
- Five decision-time features are documented with availability reasoning.
- A deliberate future-label leakage experiment is demonstrated and removed from the honest feature set.
- One named limitation is documented: GSC/GA4 availability varies by row and the warehouse only covers January 2025 through June 2026.